In [1]:
from google.colab import userdata
import os
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get('langsmith')
os.environ["LANGCHAIN_PROJECT"] = "Email Assistant"

In [2]:
!pip install -U langchain-google-genai -q
from langchain_google_genai import ChatGoogleGenerativeAI

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.2 MB/s eta 0:00:00


In [3]:
API_key = userdata.get('gemini')

llm_model = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] =  API_key
model = ChatGoogleGenerativeAI(
    model=llm_model,
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# 📧 Email Agent Overview

## 🧠 Core Idea
An intelligent email agent that adapts its behavior based on user authentication and state.

---

## 🔑 Features

- Authenticates user  
- Only allows access to the "inbox" after successful authentication  

---

## 🔄 Dynamic Behavior

- Uses **dynamic tools and prompts**
- Behavior changes based on:
  - Presence of email & password in state
  - Matching credentials with hardcoded values  

---

## 📥 Inbox Access

- Checks the "inbox" (via tool)
- Retrieves emails using a dedicated tool  

---

## 📤 Email Actions

- Can draft and send emails  
- Sending emails requires **human approval**  

---

## 🛑 Human-in-the-Loop

- Before sending an email:
  - Agent pauses execution  
  - Waits for user approval  
  - Sends only if approved  

---

## 🎯 Summary

- 🔒 Authentication controls access  
- 🔄 Dynamic middleware controls behavior  
- 🧰 Tools handle email operations  
- 👤 Human approval ensures safe execution  

In [5]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [6]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [7]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie,
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True,
            "messages": [ToolMessage(
                "Successfully authenticated",
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed",
                tool_call_id=runtime.tool_call_id)]
        })

In [9]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")

    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools)
    return handler(request)

In [10]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [11]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    model,
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call,
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )

In [15]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

/usr/local/lib/python3.12/dist-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


[{'type': 'text', 'text': 'I have successfully authenticated you. What would you like to draft?', 'extras': {'signature': 'Cn0Bvj72+0aKw0zP1A9CwlVDYKX6X5uUVcu7GvvHlgiHCMv0OAUEVySGZN81LUBCM3Hfl6VbrAx03nNUJqz2yFpMtSPmXvrHmJV4LF2TSbI1+O4U6DwXTEo2AQqmjbV7TrgYfymN9BhYMIMxrawg/w78xOmWSEe/I0h6PTxHjg=='}}]


In [16]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ),
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': 'I have successfully authenticated you. What would you like to draft?', 'extras': {'signature': 'Cn0Bvj72+0aKw0zP1A9CwlVDYKX6X5uUVcu7GvvHlgiHCMv0OAUEVySGZN81LUBCM3Hfl6VbrAx03nNUJqz2yFpMtSPmXvrHmJV4LF2TSbI1+O4U6DwXTEo2AQqmjbV7TrgYfymN9BhYMIMxrawg/w78xOmWSEe/I0h6PTxHjg=='}}]
